# 057 — final ensemble

Offline training and test prediction on **two T4 GPUs**. Attach the competition data and the private dataset made from `final.zip`; select **Internet Off**, then **Save & Run All**.

The notebook builds its teacher and train-derived preprocessing, trains the visual sources, SigLIP pooling/PCA/readout, union heads and counters, then applies the fixed 057 mixture and four-view R50 TTA. Only original ImageNet/SigLIP pretrained weights are supplied. Validation and batch calibration are disabled; epochs, batches and seeds are fixed in `configs/066_final.yaml`.

Outputs: `final_submission.csv` and a small `run.json` with actual runtime and library versions.


In [ ]:
import time
STARTED = time.monotonic()


In [ ]:
from pathlib import Path
import hashlib, json, os, sys, zipfile
import torch

input_root = Path(os.environ.get("KAGGLE_INPUT_ROOT", "/kaggle/input"))
work_root = Path(os.environ.get("KAGGLE_WORK_ROOT", "/kaggle/working"))
projects = sorted({p.parent for p in input_root.rglob("final.json")
                   if (p.parent / "src").is_dir() and (p.parent / "configs").is_dir()})
assert len(projects) <= 1, f"Several final datasets found: {projects}"
if projects:
    project = projects[0]
else:
    archives = list(input_root.rglob("final.zip"))
    assert len(archives) == 1, "Attach the private final.zip dataset"
    project = work_root / "project"
    project.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        for name in archive.namelist():
            assert (project / name).resolve().is_relative_to(project.resolve()), "Unsafe archive path"
        archive.extractall(project)
spec = json.loads((project / "final.json").read_text())
for name, digest in spec["code"].items():
    path = (project / name).resolve()
    assert path.is_relative_to(project.resolve()), "Unsafe source path"
    assert hashlib.sha256(path.read_bytes()).hexdigest() == digest, f"Source checksum mismatch: {name}"
roots = sorted({p.parent for p in input_root.rglob("train.csv")
                if (p.parent / "train").is_dir() and (p.parent / "test").is_dir()
                and (p.parent / "test.csv").is_file()
                and (p.parent / "sample_submission.csv").is_file()})
assert len(roots) == 1, f"Attach one competition dataset: {roots}"
data = roots[0]
assets = project / "pretrained"
assert torch.cuda.device_count() == 2, "Select GPU T4 x2"
sys.path.insert(0, str(project))
os.environ.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1")
print("Architecture: 057_quality | GPU: T4 x2 | Fit: train only | Validation: off")


## Train and predict

Temporary images, features and freshly trained weights stay in `/kaggle/temp`. Each restart creates a new run; no trained checkpoint or saved prediction is read from the input dataset. The final mixture is defined by `configs/057_quality.yaml`.


In [ ]:
from src.final import run
report = run(project, data, assets, Path("/kaggle/temp/final057"), work_root, started=STARTED)
print(f"Total: {report['seconds'] / 60:.2f} minutes; within 50 minutes: {report['within_50_minutes']}")
